# Notebook 10 — 行业中性化验证

**研究目标**：验证行业中性化处理对因子 IC/ICIR 的提升效果，判断是否应将中性化纳入标准因子流程。

**研究框架**：
1. 数据准备：沪深300成分股价格、收益率、合成行业/市值数据
2. 构建四个因子（动量、EP、BP、低波动）并应用行业中性化
3. 中性化前后 IC/ICIR 对比
4. 可视化改善效果
5. 结论：中性化是否值得纳入策略流程（ICIR 提升 > 10%）

> ⚠️ **行业数据说明（Tier 1 降级）**：`utils/local_data_loader.py` 不存在，改用 `utils.data_loader`。
> 真实行业分类（申万一级）需要网络调用，改用**合成行业分组**（按股票代码范围分 10 组）
> 和合成流通市值演示中性化流程。真实使用时替换为 `get_industry_classification()` + akshare 市值数据。

## 1. 数据准备

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

# 项目模块（注：utils/local_data_loader.py 不存在，使用 utils.data_loader）
from utils import (
    get_index_components,
    build_price_matrix,
    build_return_matrix,
    compute_ic_series,
    ic_summary,
    neutralize_factor,
)

# 中文字体配置
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print('✅ 模块加载完成')

In [ ]:
# 参数设置
START = '2020-01-01'
END   = '2023-12-31'
FWD_DAYS = 5  # IC 用 5 日前瞻收益

# 获取沪深300成分股
hs300 = get_index_components('000300')
symbols = hs300['symbol'].tolist()[:50]  # 取前50只，加快演示速度
print(f'股票数量: {len(symbols)}')

# 构建价格宽表与收益率宽表
price_wide = build_price_matrix(symbols, START, END, col='close', max_workers=4)
ret_wide   = build_return_matrix(price_wide)

# 5 日前瞻收益（用于 IC 计算）
fwd_ret = price_wide.pct_change(FWD_DAYS).shift(-FWD_DAYS)

# 数据质量门
assert price_wide.shape[0] > 100, f'数据行数异常: {price_wide.shape[0]}'
assert price_wide.isnull().mean().max() < 0.5, '缺失值过多'
assert price_wide.index.is_monotonic_increasing, '日期未排序'

print(f'✅ 数据质量 OK | 行数: {price_wide.shape[0]} | 时间: {price_wide.index[0].date()} ~ {price_wide.index[-1].date()}')
print(f'价格宽表: {price_wide.shape} | 收益率宽表: {ret_wide.shape}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# 构造合成行业 + 市值信息表（df_info）
# 真实使用时应替换为 get_industry_classification() + akshare 市值数据
# ─────────────────────────────────────────────────────────────

def build_synthetic_df_info(price_wide: pd.DataFrame, n_industries: int = 10) -> pd.DataFrame:
    """
    构造合成的行业/市值信息表，用于演示 neutralize_factor()。

    真实场景应使用 akshare / 本地行情库提供的申万行业分类和流通市值。

    参数:
        price_wide   : 价格宽表 (date × symbol)
        n_industries : 合成行业数量，默认10

    返回:
        df_info : 长表，列为 trade_date, symbol, ind_code, mv_float
    """
    syms = price_wide.columns.tolist()
    dates = price_wide.index

    # 按股票代码数字大小分到 n_industries 个合成行业
    sym_nums = [int(s) for s in syms]
    sym_min, sym_max = min(sym_nums), max(sym_nums)
    bins = np.linspace(sym_min - 1, sym_max + 1, n_industries + 1)
    ind_map = {}
    for s, num in zip(syms, sym_nums):
        bucket = int(np.searchsorted(bins, num, side='right') - 1)
        bucket = min(bucket, n_industries - 1)
        ind_map[s] = f'IND_{bucket:02d}'

    # 合成流通市值：用价格 * 随机股本（固定种子，保持一致性）
    rng = np.random.default_rng(42)
    shares = pd.Series(
        rng.uniform(1e8, 1e10, size=len(syms)),
        index=syms
    )

    rows = []
    for date in dates:
        for sym in syms:
            price = price_wide.loc[date, sym]
            if np.isnan(price):
                continue
            rows.append({
                'trade_date': date,
                'symbol': sym,
                'ind_code': ind_map[sym],
                'mv_float': price * shares[sym],
            })

    return pd.DataFrame(rows)


df_info = build_synthetic_df_info(price_wide, n_industries=10)
print(f'df_info 形状: {df_info.shape}')
print(f'行业分布:\n{df_info.groupby("ind_code")["symbol"].nunique().to_string()}')
print('\n⚠️  以上为合成行业分组，真实研究需替换为 akshare 申万行业分类')

## 2. 构建因子并应用行业中性化

计算四个因子的原始版本与行业中性化版本：
- **动量因子**：20日价格动量（价格涨幅）
- **EP因子**：1/PE（盈利收益率代理，用价格倒数模拟）
- **BP因子**：1/PB（账面市值比代理，用价格波动率反向模拟）
- **低波动因子**：过去60日收益率标准差的负数

In [ ]:
# ─────────────────────────────────────────────────────────────
# 原始因子构建
# 注：EP/BP 通常需要财务数据，此处用价格数据代理演示中性化效果
# ─────────────────────────────────────────────────────────────

# 1. 动量因子：20日价格动量（skip=1 避免短期反转噪音）
momentum_raw = price_wide.pct_change(20).shift(1)

# 2. EP因子代理：用「价格水平倒数」模拟（价格低 ≈ 便宜股票）
#    真实 EP = earnings / price，需财务数据
ep_raw = (1.0 / price_wide).shift(1)

# 3. BP因子代理：用「60日波动率倒数」模拟（低波动 ≈ 价值股特征）
#    真实 BP = book_value / market_cap，需财务数据
rolling_std = ret_wide.rolling(60, min_periods=20).std()
bp_raw = (1.0 / rolling_std).shift(1)

# 4. 低波动因子：60日收益率标准差负数（越低越好）
lowvol_raw = (-rolling_std).shift(1)

raw_factors = {
    '动量(20日)': momentum_raw,
    'EP代理(1/Price)': ep_raw,
    'BP代理(1/Vol)': bp_raw,
    '低波动(-Vol)': lowvol_raw,
}

for name, fac in raw_factors.items():
    valid = fac.notna().sum().sum()
    total = fac.shape[0] * fac.shape[1]
    print(f'{name}: 有效值 {valid}/{total} ({valid/total:.1%})')

In [ ]:
# ─────────────────────────────────────────────────────────────
# 应用行业中性化
# neutralize_factor(factor_wide, df_info, n_sigma=3.0)
#   - 每截面：去极值（±3σ）→ OLS 回归（行业哑变量+对数市值）→ 取残差
# ─────────────────────────────────────────────────────────────

neutral_factors = {}

for name, fac in raw_factors.items():
    print(f'正在中性化: {name} ...')
    neutral_fac = neutralize_factor(fac, df_info, n_sigma=3.0)
    neutral_factors[f'{name}_中性化'] = neutral_fac
    print(f'  完成，形状: {neutral_fac.shape}')

print('\n✅ 所有因子中性化完成')

## 3. IC/ICIR 对比分析

分别计算原始因子和中性化因子的 IC 序列，汇总 IC 均值、ICIR、t 统计量。

In [ ]:
def compute_ic_stats(fac: pd.DataFrame, ret: pd.DataFrame, name: str) -> dict:
    """计算因子的 IC 统计量，返回摘要字典。"""
    ic_s = compute_ic_series(fac, ret, method='spearman')
    stats = ic_summary(ic_s, name=name)
    stats['ic_series'] = ic_s
    return stats


# 计算原始因子 IC
print('=' * 60)
print('原始因子 IC 统计')
print('=' * 60)
raw_stats = {}
for name, fac in raw_factors.items():
    raw_stats[name] = compute_ic_stats(fac, fwd_ret, name=name)

print('\n' + '=' * 60)
print('中性化因子 IC 统计')
print('=' * 60)
neutral_stats = {}
for name, fac in neutral_factors.items():
    neutral_stats[name] = compute_ic_stats(fac, fwd_ret, name=name)

In [ ]:
# ─────────────────────────────────────────────────────────────
# 构建对比表
# ─────────────────────────────────────────────────────────────

factor_names = list(raw_factors.keys())

rows = []
for fname in factor_names:
    r = raw_stats[fname]
    n = neutral_stats[f'{fname}_中性化']

    icir_raw = r['ICIR'] if not np.isnan(r['ICIR']) else 0.0
    icir_neu = n['ICIR'] if not np.isnan(n['ICIR']) else 0.0

    icir_improvement = (abs(icir_neu) - abs(icir_raw)) / (abs(icir_raw) + 1e-9) * 100

    rows.append({
        '因子': fname,
        'IC均值_原始': round(r['IC_mean'], 4),
        'ICIR_原始': round(icir_raw, 4),
        '|t|_原始': round(abs(r['t_stat']), 3),
        'IC均值_中性化': round(n['IC_mean'], 4),
        'ICIR_中性化': round(icir_neu, 4),
        '|t|_中性化': round(abs(n['t_stat']), 3),
        'ICIR提升%': round(icir_improvement, 1),
        '推荐中性化': '✅ 是' if icir_improvement > 10 else '❌ 否',
    })

comparison_df = pd.DataFrame(rows)

print('\n行业中性化效果对比表')
print('=' * 100)
print(comparison_df.to_string(index=False))
print()
print('评判标准：ICIR 提升 > 10% 则推荐中性化')

## 4. 可视化改善效果

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

colors_raw     = '#5B9BD5'
colors_neutral = '#ED7D31'

for i, fname in enumerate(factor_names):
    ax = axes[i]
    r = raw_stats[fname]
    n = neutral_stats[f'{fname}_中性化']

    icir_raw = r['ICIR'] if not np.isnan(r['ICIR']) else 0.0
    icir_neu = n['IC_mean'] / n['IC_std'] if n['IC_std'] > 0 else 0.0

    # IC 均值 + ICIR 并排柱状图
    metrics = ['IC均值', 'ICIR']
    raw_vals     = [r['IC_mean'], icir_raw]
    neutral_vals = [n['IC_mean'], icir_neu]

    x = np.arange(len(metrics))
    width = 0.35

    bars1 = ax.bar(x - width/2, raw_vals,     width, label='原始因子',   color=colors_raw,     alpha=0.85)
    bars2 = ax.bar(x + width/2, neutral_vals, width, label='中性化因子', color=colors_neutral, alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(metrics, fontsize=11)
    ax.set_ylabel('数值', fontsize=10)
    ax.set_title(fname, fontsize=12, fontweight='bold')
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.legend(fontsize=9)

    # 在柱子上标值
    for bar in bars1:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.002 * np.sign(h),
                f'{h:.3f}', ha='center', va='bottom' if h >= 0 else 'top', fontsize=8)
    for bar in bars2:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.002 * np.sign(h),
                f'{h:.3f}', ha='center', va='bottom' if h >= 0 else 'top', fontsize=8)

plt.suptitle('行业中性化前后 IC均值 / ICIR 对比', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ICIR 提升百分比汇总柱状图
fig, ax = plt.subplots(figsize=(10, 5))

icir_improvements = comparison_df.set_index('因子')['ICIR提升%']
bar_colors = ['#70AD47' if v > 10 else '#FF0000' for v in icir_improvements]

bars = ax.bar(icir_improvements.index, icir_improvements.values,
              color=bar_colors, alpha=0.85, edgecolor='white')

ax.axhline(10, color='green', linewidth=1.5, linestyle='--', label='推荐阈值 +10%')
ax.axhline(0,  color='black', linewidth=0.8)

for bar, val in zip(bars, icir_improvements.values):
    ax.text(bar.get_x() + bar.get_width()/2,
            val + (0.5 if val >= 0 else -1.5),
            f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel('ICIR 提升 (%)', fontsize=11)
ax.set_title('行业中性化 ICIR 提升幅度（绿色 = 超过推荐阈值 10%）', fontsize=12)
ax.legend(fontsize=10)
ax.set_xticklabels(icir_improvements.index, rotation=15, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# IC 时序对比（以动量因子为例）
fname = '动量(20日)'
ic_raw_s     = raw_stats[fname]['ic_series']
ic_neutral_s = neutral_stats[f'{fname}_中性化']['ic_series']

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for ax, ic_s, label, color in [
    (axes[0], ic_raw_s,     '原始因子',   '#5B9BD5'),
    (axes[1], ic_neutral_s, '中性化因子', '#ED7D31'),
]:
    ic_clean = ic_s.dropna()
    rolling  = ic_clean.rolling(20).mean()
    ax.bar(ic_clean.index, ic_clean.values, alpha=0.3, color=color, width=1)
    ax.plot(rolling.index, rolling.values, color=color, linewidth=1.5, label='20日滚动均值')
    ax.axhline(0, color='black', linewidth=0.5)
    ic_mean = ic_clean.mean()
    icir    = ic_clean.mean() / ic_clean.std() if ic_clean.std() > 0 else float('nan')
    ax.set_title(f'{fname} — {label}  (IC均值={ic_mean:.4f}, ICIR={icir:.4f})', fontsize=11)
    ax.set_ylabel('IC', fontsize=10)
    ax.legend()

plt.suptitle('动量因子 IC 时序：原始 vs 行业中性化', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. 结论

根据 ICIR 提升幅度（阈值 > 10%）判断是否推荐将行业中性化纳入策略流程。

In [ ]:
print('=' * 70)
print('  行业中性化验证 — 结论汇总')
print('=' * 70)
print()

recommend_count = 0

for fname in factor_names:
    r = raw_stats[fname]
    n = neutral_stats[f'{fname}_中性化']
    icir_raw = r['ICIR'] if not np.isnan(r['ICIR']) else 0.0
    icir_neu = n['ICIR'] if not np.isnan(n['ICIR']) else 0.0
    icir_improvement = (abs(icir_neu) - abs(icir_raw)) / (abs(icir_raw) + 1e-9) * 100

    verdict = '✅ 推荐中性化' if icir_improvement > 10 else '❌ 中性化无明显提升'
    print(f'  {fname:<20} ICIR: {icir_raw:+.4f} → {icir_neu:+.4f}  (提升 {icir_improvement:+.1f}%)  {verdict}')
    if icir_improvement > 10:
        recommend_count += 1

print()
print('-' * 70)
pct = recommend_count / len(factor_names) * 100
if recommend_count >= len(factor_names) // 2:
    print(f'  总体结论：{recommend_count}/{len(factor_names)} 个因子受益于中性化（{pct:.0f}%）')
    print('  ➡️  建议将行业中性化纳入标准因子预处理流程')
else:
    print(f'  总体结论：仅 {recommend_count}/{len(factor_names)} 个因子受益（{pct:.0f}%）')
    print('  ➡️  行业中性化整体提升有限，可视因子情况选择性使用')

print()
print('分析要点：')
print('  1. 行业中性化通过去除行业beta暴露，使因子信号更加纯粹')
print('  2. 对行业轮动明显的时期，中性化效果更显著')
print('  3. 本结论基于合成行业分组，真实研究需使用申万一级行业分类')
print('  4. EP/BP 因子基于价格代理，真实财务数据可能产生不同结论')
print('  5. 下一步：接入 get_industry_classification() 验证真实行业数据效果')
print('=' * 70)